# WIC Edge Case Analysis

Deep dive into the policy boundaries that trip up LLMs when reasoning about WIC:
- 185% FPL income limit (vs. 130% for SNAP gross)
- No asset test
- Categorical eligibility via SNAP, Medicaid, or TANF
- Participant category requirement (pregnant, breastfeeding, postpartum, infant, child under 5)
- WIC runs July 1–June 30, not the federal fiscal year

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from govsynth.sources.us.wic import WICSource
from govsynth.sources.us.snap import SNAPSource
from govsynth.profiles.us_household import USHouseholdProfile
from govsynth import Pipeline

print('Ready.')

## 1. Introduction — WIC vs SNAP: What's Different

WIC and SNAP are both USDA nutrition programs, but their eligibility rules diverge significantly:

| Feature | SNAP | WIC |
|---|---|---|
| Income limit | 130% FPL gross / 100% FPL net | **185% FPL gross only** |
| Asset test | Yes (varies by state/BBCE) | **None** |
| Categorical eligibility | BBCE-dependent | **SNAP/Medicaid/TANF = auto income-eligible** |
| Net income deductions | Complex (earned, standard, shelter) | **Not applicable** |
| Participant categories | All household members | **Pregnant, breastfeeding, postpartum women; infants; children under 5** |
| Benefit type | Cash-equivalent EBT | Food package + nutrition education |
| Program year | Federal FY (Oct 1–Sep 30) | **WIC IEG year (Jul 1–Jun 30)** |
| Nutritional risk | Not required | **Required — determined at WIC clinic** |

**Key LLM failure modes for WIC:**
1. Applying a 130% FPL limit instead of 185%
2. Imposing an asset test that doesn't exist
3. Missing the categorical eligibility shortcut
4. Omitting the nutritional risk caveat
5. Misidentifying participant categories (e.g., a father in the household is not a WIC participant)

## 2. Inspect WIC FY2026 Income Limits

Load the WIC FY2026 threshold table and compare it to SNAP FY2026 gross limits for the same household sizes.

In [ ]:
source_wic = WICSource(fiscal_year=2026, state='VA')
source_snap = SNAPSource(fiscal_year=2026, state='VA')

t_wic = source_wic.thresholds()
t_snap = source_snap.thresholds()

print(f'WIC period: {t_wic.extra["effective_start"]} to {t_wic.extra["effective_end"]}')
print(f'WIC income limit: {t_wic.extra["income_limit_pct_fpl"]}% FPL  |  CFR: {t_wic.extra["cfr_reference"]}')
print(f'No asset test: asset_limit_general = {t_wic.asset_limit_general}')
print()
print(f'{"HH Size":<10} {"WIC Limit (185% FPL)":<25} {"SNAP Gross (130% FPL)":<25} {"WIC Premium"}')
print('-' * 70)
for size in range(1, 9):
    wic_lim = t_wic.by_household_size(size).gross_monthly
    snap_lim = t_snap.by_household_size(size).gross_monthly
    premium = wic_lim - snap_lim
    print(f'{size:<10} ${wic_lim:>10,.2f}/mo          ${snap_lim:>10,.2f}/mo          +${premium:,.2f}')

In [ ]:
# Show eligible participant categories and categorical eligibility programs
print('WIC eligible participant categories:')
for cat in t_wic.extra['eligible_categories']:
    print(f'  - {cat}')

print()
print('Categorical eligibility programs (auto income-eligible):')
for prog in t_wic.extra['categorical_eligibility_programs']:
    print(f'  - {prog.upper()}')

## 3. 185% FPL Income Boundary

Sweep income offsets around the 185% FPL limit for a 3-person household. This is the primary eligibility gate and the most common LLM error point.

In [ ]:
hh_size = 3
limits_3 = t_wic.by_household_size(hh_size)

print(f'3-person WIC income limit (185% FPL): ${limits_3.gross_monthly:,.2f}/month')
print(f'Period: {t_wic.extra["effective_start"]} – {t_wic.extra["effective_end"]}')
print()

offsets = [
    ('10% below', -0.10),
    (' 1% below', -0.01),
    ('AT LIMIT',   0.00),
    (' 1% above',  0.01),
    ('10% above',  0.10),
]

print(f'{"Label":<12} {"Monthly Gross":<18} {"Result":<14} Reason')
print('-' * 85)
for label, pct in offsets:
    profile = USHouseholdProfile.at_threshold(
        program='wic', threshold='income_limit_185pct_fpl',
        state='VA', household_size=hh_size, fiscal_year=2026,
        offset_pct=pct, seed=42,
    )
    eligible, reason = source_wic.is_eligible(
        household_size=profile.household_size,
        monthly_gross_income=profile.monthly_gross_income,
        participant_category='pregnant',
    )
    result = 'ELIGIBLE' if eligible else 'INELIGIBLE'
    print(f'{label:<12} ${profile.monthly_gross_income:>10,.2f}       {result:<14} {reason[:55]}')

In [ ]:
# Confirm: no net income test or asset test applies
limit = limits_3.gross_monthly
test_gross = round(limit * 1.01, 2)  # 1% above — ineligible on income
large_assets = 500_000  # Enormous assets — irrelevant for WIC

eligible, reason = source_wic.is_eligible(
    household_size=hh_size,
    monthly_gross_income=test_gross,
    participant_category='pregnant',
    is_categorically_eligible=False,
)
print(f'Income ${test_gross:,.2f} (1% above limit), assets=${large_assets:,}')
print(f'Result: {"ELIGIBLE" if eligible else "INELIGIBLE"}')
print(f'Reason: {reason}')
print()
print('Note: WIC has NO asset test. The $500,000 in assets has zero effect on eligibility.')
print('The only income test is gross income vs. 185% FPL. (7 CFR 246.7(d)(1))')

## 4. Household Size Sensitivity

A fixed income level can be eligible or ineligible depending purely on household size. This matters when household composition changes (e.g., a new infant is added to the household, increasing both household size and the WIC limit).

In [ ]:
# Fix income at the 3-person limit — show how it fares across all household sizes
fixed_income = t_wic.by_household_size(3).gross_monthly  # $4,109

print(f'Fixed monthly gross income: ${fixed_income:,.2f}')
print(f'{"HH Size":<10} {"WIC Limit":<18} {"Income vs. Limit":<20} {"Result"}')
print('-' * 60)
for size in range(1, 9):
    lim = t_wic.by_household_size(size).gross_monthly
    eligible, _ = source_wic.is_eligible(
        household_size=size,
        monthly_gross_income=fixed_income,
        participant_category='pregnant',
    )
    pct_of_limit = fixed_income / lim * 100
    result = 'ELIGIBLE' if eligible else 'INELIGIBLE'
    marker = ' <-- flip point' if size == 3 else ''
    print(f'{size:<10} ${lim:>10,.2f}     {pct_of_limit:>6.1f}% of limit    {result}{marker}')

In [ ]:
# Practical scenario: infant joins the household
# A 2-person household (pregnant woman + partner) has income just above the 2-person limit.
# After the infant is born, the household grows to 3 — and the new infant is itself a WIC participant.

income = 3400.0  # Between 2-person ($3,261) and 3-person ($4,109) WIC limits

elig_2, reason_2 = source_wic.is_eligible(
    household_size=2, monthly_gross_income=income, participant_category='pregnant'
)
elig_3, reason_3 = source_wic.is_eligible(
    household_size=3, monthly_gross_income=income, participant_category='infant'
)

print(f'Income: ${income:,.2f}/month')
print()
print(f'During pregnancy (2-person HH): {"ELIGIBLE" if elig_2 else "INELIGIBLE"}')
print(f'  {reason_2}')
print()
print(f'After birth — infant in 3-person HH: {"ELIGIBLE" if elig_3 else "INELIGIBLE"}')
print(f'  {reason_3}')
print()
print('Key insight: the infant\'s birth increases household size, raising the WIC income limit.')
print('A family ineligible during pregnancy may become eligible once the infant is added to the count.')

## 5. Categorical Eligibility

Households already enrolled in SNAP, Medicaid, or TANF are **automatically income-eligible** for WIC — the 185% FPL income test is waived entirely. This is one of the most commonly missed facts in LLM WIC reasoning.

Note: categorical eligibility waives the *income* test only. The applicant must still:
1. Be in a WIC-eligible participant category (pregnant, breastfeeding, postpartum, infant, child under 5)
2. Receive a nutritional risk determination at a WIC clinic

In [ ]:
# A household with income FAR above the 185% FPL limit
# Normally ineligible — but they receive Medicaid
hh_size = 2
wic_limit_2 = t_wic.by_household_size(2).gross_monthly  # $3,261
high_income = round(wic_limit_2 * 2.0, 2)  # 200% of the WIC limit — double the threshold

# Without categorical eligibility
elig_income, reason_income = source_wic.is_eligible(
    household_size=hh_size,
    monthly_gross_income=high_income,
    participant_category='breastfeeding',
    is_categorically_eligible=False,
)

# With categorical eligibility (Medicaid)
elig_cat, reason_cat = source_wic.is_eligible(
    household_size=hh_size,
    monthly_gross_income=high_income,
    participant_category='breastfeeding',
    is_categorically_eligible=True,
)

print(f'2-person WIC income limit: ${wic_limit_2:,.2f}/month')
print(f'Test household income:     ${high_income:,.2f}/month  ({high_income/wic_limit_2*100:.0f}% of WIC limit)')
print(f'Participant category:      breastfeeding')
print()
print('Without categorical eligibility (income-only test):')
print(f'  Result: {"ELIGIBLE" if elig_income else "INELIGIBLE"}')
print(f'  Reason: {reason_income}')
print()
print('With categorical eligibility (enrolled in Medicaid):')
print(f'  Result: {"ELIGIBLE" if elig_cat else "INELIGIBLE"}')
print(f'  Reason: {reason_cat}')

In [ ]:
# Show that categorical eligibility does NOT override the participant category check
elig_invalid_cat, reason_invalid_cat = source_wic.is_eligible(
    household_size=2,
    monthly_gross_income=high_income,
    participant_category='adult_male',  # Not a valid WIC category
    is_categorically_eligible=True,
)

print('Categorical eligibility does NOT bypass the participant category requirement:')
print(f'  participant_category="adult_male", is_categorically_eligible=True')
print(f'  Result: {"ELIGIBLE" if elig_invalid_cat else "INELIGIBLE"}')
print(f'  Reason: {reason_invalid_cat}')
print()
print('Valid WIC participant categories:', t_wic.extra['eligible_categories'])

## 6. Generate WIC Test Cases

Use the `Pipeline.from_preset('wic.national')` to generate a batch of WIC eligibility test cases. WIC cases include categorical eligibility scenarios, boundary income cases, and multiple participant category types.

In [ ]:
pipeline = Pipeline.from_preset('wic.national')
cases = pipeline.generate(n=20, seed=42)

print(f'Generated: {len(cases)} WIC test cases')
print(f'Program:   {cases[0].program.value if cases else "N/A"}')
print()

In [ ]:
from collections import Counter

# Outcome distribution
outcomes = Counter(c.expected_outcome for c in cases)
print('Outcome distribution:')
for outcome, n in sorted(outcomes.items()):
    pct = n / len(cases) * 100
    bar = '\u2588' * int(pct / 4)
    print(f'  {str(outcome):<14} {n:>3}  {pct:>5.1f}%  {bar}')

print()

# Difficulty distribution
diff = Counter(c.difficulty.value for c in cases)
print('Difficulty distribution:')
for d, n in sorted(diff.items()):
    pct = n / len(cases) * 100
    bar = '\u2588' * int(pct / 4)
    print(f'  {d:<14} {n:>3}  {pct:>5.1f}%  {bar}')

print()

# Profile strategy distribution
strategies = Counter(c.metadata.get('profile_strategy', 'unknown') for c in cases)
print('Profile strategy distribution:')
for strat, n in sorted(strategies.items()):
    print(f'  {strat:<35} {n}')

In [ ]:
# Quick tabular summary of all cases
print(f'{"#":<3} {"Case ID":<50} {"Outcome":<12} {"Difficulty"}')
print('-' * 85)
for i, case in enumerate(cases, 1):
    outcome = str(case.expected_outcome)
    diff = case.difficulty.value
    print(f'{i:<3} {case.case_id:<50} {outcome:<12} {diff}')

## 7. Inspect a Case

Look at a complete generated test case — scenario, task, and rationale trace. Categorical eligibility cases are particularly illustrative because the rationale must explain the income waiver.

In [ ]:
# Pick a categorically eligible case if available, otherwise the first case
cat_cases = [
    c for c in cases
    if 'categorical' in c.case_id or 'categorical' in c.metadata.get('profile_strategy', '')
]
case = cat_cases[0] if cat_cases else cases[0]

print(f'Case ID:    {case.case_id}')
print(f'Outcome:    {case.expected_outcome}')
print(f'Difficulty: {case.difficulty.value}')
print(f'Strategy:   {case.metadata.get("profile_strategy", "N/A")}')
print()
print('Scenario:')
print(f'  {case.scenario.summary}')
if case.scenario.details:
    print()
    for line in case.scenario.details:
        print(f'  {line}')

In [ ]:
print('Rationale Trace:')
print('=' * 60)
print(case.rationale_trace.to_plain_text())

In [ ]:
# Also inspect a hard income-boundary case for contrast
hard_cases = [
    c for c in cases
    if c.difficulty.value == 'hard' and 'at_limit' in c.case_id
]
if not hard_cases:
    hard_cases = [c for c in cases if c.difficulty.value == 'hard']

if hard_cases:
    h = hard_cases[0]
    print(f'Hard boundary case: {h.case_id}')
    print(f'Outcome: {h.expected_outcome}')
    print()
    print('Rationale Trace:')
    print('=' * 60)
    print(h.rationale_trace.to_plain_text())
else:
    print('No hard cases found in this batch — try a larger n or different seed.')